## Cell 1 — Install dependencies

In [ ]:
!pip install -q gatspy astropy astroquery scipy numpy pandas matplotlib tqdm

## Cell 2 — Clone repo and add src to path

In [ ]:
import os, sys

REPO = "/content/asteroid-pipeline"
if not os.path.exists(REPO):
    os.system(f"git clone https://github.com/wonrobot/asteroid-pipeline.git {REPO}")
else:
    os.system(f"cd {REPO} && git pull")

sys.path.insert(0, f"{REPO}/src")
print("Repo ready.")

## Cell 3 — Upload CSV

In [ ]:
from google.colab import files
uploaded = files.upload()   # select bq-results-today.csv
CSV_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {CSV_PATH}")

## Cell 4 — Load and inspect data

In [ ]:
import pandas as pd
import numpy as np
from config import PipelineConfig, DataConfig, PeriodConfig, TierConfig, OutputConfig

# The CSV has bands already in Lg/Lr/Li/Lu format (not raw g/r/i).
# Override bands_use to match — otherwise ingestion filters everything out.
config = PipelineConfig(
    data=DataConfig(
        bands_use=["Lg", "Lr", "Li"],  # pre-remapped names in this CSV
        band_remap={},                  # no remapping needed
        rmsmag_max=0.21,
        min_obs_total=20,
        min_obs_band=5,
    ),
    period=PeriodConfig(),
    tier=TierConfig(mbls_fap_n_perm=200),
    output=OutputConfig(
        results_dir="/content/results",
        catalog_file="/content/results/validation_catalog.csv",
        log_file="/content/results/validation.log",
        verbose=True,
    ),
)

# Load — tab-separated, obstime with timezone
df_raw = pd.read_csv(CSV_PATH, sep="\t", parse_dates=["obstime"])
print(f"Raw rows: {len(df_raw):,}  |  Asteroids: {df_raw['provid'].nunique():,}")
print(f"Bands in file: {sorted(df_raw['band'].unique())}")

# Compute MJD
epoch = pd.Timestamp("1858-11-17")
df_raw["mjd"] = (
    pd.to_datetime(df_raw["obstime"], utc=True)
    .dt.tz_localize(None).subtract(epoch)
    .dt.total_seconds().div(86400.0)
)

# Quality cuts
df = df_raw[df_raw["band"].isin(config.data.bands_use)].copy()
df = df[df["rmsmag"] <= config.data.rmsmag_max]
df = df[df["rmsmag"] > 0]
df = df.dropna(subset=["mag","rmsmag","mjd","band","provid"])
df = df.sort_values(["provid","mjd"]).reset_index(drop=True)

print(f"\nAfter quality cuts: {len(df):,} rows  |  {df['provid'].nunique():,} asteroids")

from ingestion import list_objects
summary = list_objects(df)
print(f"\nTop 20 by observation count:")
print(summary.head(20).to_string(index=False))

## Cell 5 — LCDB lookup

In [ ]:
from sources.lcdb import download_lcdb, load_lcdb, lookup_batch

LCDB_CACHE = "/content/lcdb_cache.csv"
download_lcdb(LCDB_CACHE)
df_lcdb = load_lcdb(LCDB_CACHE)
print(f"LCDB loaded: {len(df_lcdb):,} records")

provids = df["provid"].unique().tolist()
lcdb_records = lookup_batch(provids, df_lcdb)

found     = [(p,r) for p,r in lcdb_records.items() if r.found]
not_found = [p for p,r in lcdb_records.items() if not r.found]
print(f"\nLCDB matches: {len(found)} / {len(provids)}")

if found:
    print("\nKnown periods (U>=2):")
    for provid, rec in found:
        if rec.u_code >= 2:
            print(f"  {provid:20s}  P={rec.period_hr:.3f}hr  U={rec.u_flag}")

print(f"\nNot in LCDB (new/recent): {len(not_found)}")
if not_found:
    print("  " + ", ".join(not_found[:15]))

## Cell 6 — Run pipeline on all asteroids

In [ ]:
from pipeline import run_pipeline
os.makedirs("/content/results", exist_ok=True)

catalog = run_pipeline(df, config=config, save_every_n=50)

print(f"\nCatalog: {len(catalog)} rows")
show_cols = ["provid","final_period_hr","reliability","r_code","r_flag",
             "t2_p_value","t2_mbls_fap","t2_mbls_band_support_frac"]
print(catalog[[c for c in show_cols if c in catalog.columns]].to_string(index=False))

## Cell 7 — Validation report: pipeline vs LCDB

In [ ]:
from sources.lcdb import compare_to_lcdb

rows = []
for _, row in catalog.iterrows():
    provid = row["provid"]
    rec    = lcdb_records.get(provid)
    pipe_p = row.get("final_period_hr", np.nan)

    entry = {
        "provid":      provid,
        "n_obs":       row.get("t1_n_obs"),
        "pipe_period": pipe_p,
        "reliability": row.get("reliability"),
        "r_code":      row.get("r_code"),
        "r_flag":      row.get("r_flag"),
        "t1_passes":   row.get("t1_passes"),
        "t2_passes":   row.get("t2_passes"),
        "mhaov_p":     row.get("t2_p_value"),
        "mbls_fap":    row.get("t2_mbls_fap"),
        "band_frac":   row.get("t2_mbls_band_support_frac"),
        "lcdb_period": np.nan,
        "lcdb_u":      "",
        "lcdb_agree":  "no_prior",
        "delta_pct":   np.nan,
    }

    if rec and rec.found and not np.isnan(rec.period_hr):
        entry["lcdb_period"] = rec.period_hr
        entry["lcdb_u"]      = rec.u_flag
        if not np.isnan(pipe_p):
            cmp = compare_to_lcdb(pipe_p, rec)
            entry["lcdb_agree"] = cmp["agreement"]
            entry["delta_pct"]  = round(cmp["delta_pct"]*100, 2)
    rows.append(entry)

val = pd.DataFrame(rows)

print("=" * 70)
print("VALIDATION REPORT")
print("=" * 70)
print(f"Total processed   : {len(val)}")
print(f"Tier 1 passed     : {int(val['t1_passes'].sum())}")
print(f"Tier 2 passed     : {int(val['t2_passes'].sum())}")
print(f"Published (R>=1)  : {int(val['r_code'].ge(1).sum())}")
print(f"  R=3 high conf   : {int(val['r_code'].eq(3).sum())}")
print(f"  R=2 moderate    : {int(val['r_code'].eq(2).sum())}")
print(f"  R=1 tentative   : {int(val['r_code'].eq(1).sum())}")
print(f"Alias flagged R=-1: {int(val['r_code'].eq(-1).sum())}")
print(f"No period   R=0   : {int(val['r_code'].eq(0).sum())}")

has_lcdb = val[val["lcdb_period"].notna()]
if len(has_lcdb):
    print(f"\n{'─'*70}")
    print(f"LCDB comparison ({len(has_lcdb)} objects with known periods):")
    print(f"{'─'*70}")
    print(has_lcdb[["provid","pipe_period","lcdb_period","lcdb_u",
                     "lcdb_agree","delta_pct","r_code","r_flag"]].to_string(index=False))

new_pub = val[val["lcdb_period"].isna() & val["r_code"].ge(1)]
if len(new_pub):
    print(f"\n{'─'*70}")
    print(f"NEW detections (not in LCDB): {len(new_pub)}")
    print(f"{'─'*70}")
    print(new_pub[["provid","pipe_period","r_code","r_flag",
                   "mhaov_p","mbls_fap","band_frac","n_obs"]].to_string(index=False))

failed = val[val["t1_passes"]==False]
if len(failed):
    print(f"\n{'─'*70}")
    print(f"Tier 1 rejections: {len(failed)}")
    for _, r in failed.iterrows():
        cat_row = catalog[catalog["provid"]==r["provid"]]
        reason  = cat_row["t1_reject_reason"].iloc[0] if len(cat_row) else ""
        print(f"  {r['provid']:20s}  n_obs={int(r['n_obs']) if r['n_obs'] else '?'}  {reason}")

## Cell 8 — Phase-folded lightcurve plots

In [ ]:
import matplotlib.pyplot as plt

BAND_COLORS = {"Lg":"#2196F3","Lr":"#F44336","Li":"#FF9800","Lu":"#9C27B0"}
R_COLORS    = {3:"#2e7d32", 2:"#1565c0", 1:"#e65100", 0:"#b71c1c", -1:"#6a1b9a"}

def plot_folded(ax, df_obj, period_hr, title, r_code=None, lcdb_p=None):
    t0    = df_obj["mjd"].min()
    t     = (df_obj["mjd"].values - t0) * 24.0
    phase = (t % period_hr) / period_hr
    for band in sorted(df_obj["band"].unique()):
        m = df_obj["band"]==band
        c = BAND_COLORS.get(band,"gray")
        ax.errorbar(phase[m],   df_obj["mag"].values[m], yerr=df_obj["rmsmag"].values[m],
                    fmt="o", ms=3, alpha=0.8, color=c, label=band, capsize=0)
        ax.errorbar(phase[m]+1, df_obj["mag"].values[m], yerr=df_obj["rmsmag"].values[m],
                    fmt="o", ms=3, alpha=0.25,color=c, capsize=0)
    ax.invert_yaxis()
    ax.set_xlabel("Phase"); ax.set_ylabel("Mag")
    lbl = title
    if lcdb_p: lbl += f"\n[LCDB={lcdb_p:.3f}hr]"
    ax.set_title(f"{lbl}  P={period_hr:.3f}hr  R={r_code}",
                 color=R_COLORS.get(r_code,"k"), fontsize=8)
    ax.legend(fontsize=7); ax.set_xlim(0,2)

targets = (val[val["r_code"].ge(1) & val["pipe_period"].notna()]
           .sort_values("r_code", ascending=False).head(12))

if len(targets)==0:
    print("No published periods to plot.")
else:
    ncols = 3
    nrows = int(np.ceil(len(targets)/ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
    axes = np.array(axes).flatten()
    for i, (_, row) in enumerate(targets.iterrows()):
        df_obj   = df[df["provid"]==row["provid"]].copy()
        lcdb_rec = lcdb_records.get(row["provid"])
        lcdb_p   = lcdb_rec.period_hr if (lcdb_rec and lcdb_rec.found) else None
        plot_folded(axes[i], df_obj, row["pipe_period"],
                    row["provid"], int(row["r_code"]), lcdb_p)
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    plt.suptitle("Phase-folded lightcurves (by R-code)", fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig("/content/results/folded_lightcurves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: /content/results/folded_lightcurves.png")

## Cell 9 — Deep dive: single asteroid

Change `INSPECT_PROVID` to any asteroid in your dataset.

In [ ]:
INSPECT_PROVID = "2025 AG16"   # <-- change this

from ingestion import load_single_object
from preprocessing import preprocess
from tier1 import run_tier1
from tier2 import run_tier2
from characterise import characterise

df_obj = load_single_object(INSPECT_PROVID, df)
data   = preprocess(df_obj, config)
char   = characterise(df_obj)
t1     = run_tier1(data, config)

print(f"{'='*60}\n{INSPECT_PROVID}\n{'='*60}")
print(f"N={data.n_obs}  bands={data.band_counts}  baseline={data.baseline_hr:.1f}hr")
print(f"SNR={data.snr:.2f}  regime={char.regime}  ceiling={char.reliability_ceiling}")
print(f"T1: passes={t1.passes}  GLS={t1.best_period_gls:.3f}hr  MBLS={t1.best_period_mbls:.3f}hr")
print(f"    gls_cont={t1.gls_contamination:.2f}  mbls_cont={t1.mbls_contamination:.2f}")

t2 = None
if t1.passes:
    t2 = run_tier2(data, t1, config)
    print(f"T2: MHAOV={t2.best_period_mhaov:.3f}hr  MBLS={t2.best_period_mbls:.3f}hr  CE={t2.best_period_ce:.3f}hr")
    print(f"    agreement={t2.agreement}  consensus={t2.consensus_period:.3f}hr")
    print(f"    MHAOV p={t2.p_value:.2e} ({'+' if t2.mhaov_sig else '-'})  "
          f"MBLS FAP={t2.mbls_fap:.4f} ({'+' if t2.mbls_sig else '-'})  both={t2.both_sig}")
    print(f"    band_support={t2.mbls_band_support}")
    print(f"    frac={t2.mbls_band_support_frac:.2f}  n_bands={t2.mbls_n_bands_supporting}")

# 4-panel diagnostics
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle(f"{INSPECT_PROVID} — Diagnostics", fontsize=12)

ax = axes[0,0]
for band in sorted(df_obj["band"].unique()):
    m = df_obj["band"]==band
    ax.errorbar(df_obj["mjd"].values[m], df_obj["mag"].values[m],
                yerr=df_obj["rmsmag"].values[m], fmt="o", ms=3, alpha=0.7,
                label=band, color=BAND_COLORS.get(band,"gray"))
ax.invert_yaxis(); ax.set_xlabel("MJD"); ax.set_ylabel("Mag")
ax.set_title("Raw lightcurve"); ax.legend(fontsize=8)

ax = axes[0,1]
if t1.passes and len(t1.test_periods)>0:
    ax.plot(t1.test_periods, t1.gls_power, "b-", lw=0.8, alpha=0.8, label="GLS")
    wp_n = t1.window_power/(t1.window_power.max()+1e-12)*t1.gls_power.max()
    ax.fill_between(t1.test_periods, wp_n, alpha=0.2, color="orange", label="Window")
    ax.axvline(t1.best_period_gls, color="b", lw=1.5, ls="--", label=f"best={t1.best_period_gls:.3f}hr")
ax.set_xlabel("Period (hr)"); ax.set_ylabel("GLS power")
ax.set_title(f"T1 GLS (gls_cont={t1.gls_contamination:.2f})"); ax.legend(fontsize=8)

ax = axes[1,0]
if t2:
    mh_n = t2.mhaov_power/(t2.mhaov_power.max()+1e-12)
    mb_n = t2.mbls_power /(t2.mbls_power.max() +1e-12)
    ax.plot(t2.test_periods, mh_n, "g-", lw=0.8, alpha=0.8, label="MHAOV (norm)")
    ax.plot(t2.test_periods, mb_n, "r-", lw=0.8, alpha=0.8, label="MBLS (norm)")
    ax.axvline(t2.best_period_mhaov, color="g", lw=1.5, ls="--", label=f"MHAOV={t2.best_period_mhaov:.3f}hr")
    ax.axvline(t2.best_period_mbls,  color="r", lw=1.5, ls=":",  label=f"MBLS={t2.best_period_mbls:.3f}hr")
ax.set_xlabel("Period (hr)"); ax.set_ylabel("Norm power")
ax.set_title(f"T2 MHAOV+MBLS  agree={t2.agreement if t2 else 'N/A'}"); ax.legend(fontsize=8)

ax = axes[1,1]
pipe_row = val[val["provid"]==INSPECT_PROVID]
if len(pipe_row) and not np.isnan(float(pipe_row.iloc[0]["pipe_period"] or "nan")):
    p  = float(pipe_row.iloc[0]["pipe_period"])
    rc = int(pipe_row.iloc[0]["r_code"])
    lcdb_rec = lcdb_records.get(INSPECT_PROVID)
    lcdb_p   = lcdb_rec.period_hr if (lcdb_rec and lcdb_rec.found) else None
    plot_folded(ax, df_obj, p, INSPECT_PROVID, rc, lcdb_p)
else:
    ax.text(0.5,0.5,"No period published",ha="center",va="center",transform=ax.transAxes)
    ax.set_title(f"{INSPECT_PROVID} — no period")

plt.tight_layout()
fname = f"/content/results/{INSPECT_PROVID.replace(' ','_')}_diagnostics.png"
plt.savefig(fname, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fname}")

## Cell 10 — Save and download results

In [ ]:
val.to_csv("/content/results/validation_report.csv", index=False)
catalog.to_csv("/content/results/pipeline_catalog.csv", index=False)
print("Saved:\n  /content/results/validation_report.csv\n  /content/results/pipeline_catalog.csv")

from google.colab import files
files.download("/content/results/validation_report.csv")
files.download("/content/results/pipeline_catalog.csv")